# Lesson 3: Stake effects and presentation order — de Hollander et al. (2024)

## Background

De Hollander et al. (2024, *Nature Human Behaviour*) tested whether perceptual noise
during the encoding of numerical magnitudes explains risk aversion and its interaction
with presentation order.  The key design feature: the **order** in which the safe and
risky options are presented is randomised across trials, allowing the model to disentangle
$\nu_1$ (noise on the first-presented option) from $\nu_2$ (noise on the second-presented
option).

This produces a distinctive **presentation-order × stake-size interaction**: when the
safe option comes first, high safe stakes are compressed downward more strongly by the
prior → safe looks less attractive → the observer becomes more risk-seeking for high
stakes.  When the risky option comes first, the same mechanism operates on the risky
stakes → risk aversion for high stakes.

Standard models (EU, KLW with a shared noise) **cannot** capture this asymmetry.

### Models compared

| Model | Class | Key parameters |
|-------|-------|----------------|
| Expected Utility (EU) | `ExpectedUtilityRiskModel` | `alpha`, `sigma` |
| KLW | `RiskModel(prior_estimate='klw', fit_seperate_evidence_sd=False)` | `evidence_sd`, `prior_sd` (shared) |
| PMCM | `RiskModel(prior_estimate='full', fit_seperate_evidence_sd=True)` | `n1_evidence_sd`, `n2_evidence_sd`, `risky/safe_prior_mu/sd` |

We fit all three on both the **dot-cloud** (fMRI sessions 3T+7T) and the **symbolic**
(Arabic numerals) datasets, then compare posterior predictives against the interaction
pattern.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
from bauer.utils.data import load_dehollander2024
from bauer.models import ExpectedUtilityRiskModel, RiskModel

# Load both fMRI sessions and combine
df_dot = load_dehollander2024(task='dotcloud', sessions=['3t2', '7t2'])
df_sym = load_dehollander2024(task='symbolic')

print(f"Dot clouds  — subjects: {df_dot.index.get_level_values('subject').nunique()}, "
      f"trials: {len(df_dot)}")
print(f"Symbolic    — subjects: {df_sym.index.get_level_values('subject').nunique()}, "
      f"trials: {len(df_sym)}")
df_dot.head()

In [ ]:
def prep_df(df):
    """Add log_ratio, chose_risky, n_safe, order flag, and binned columns."""
    df = df.reset_index()
    risky_first = df['p1'] == 0.55
    df['log_ratio']     = np.log(
        np.where(risky_first, df['n1'], df['n2']) /
        np.where(risky_first, df['n2'], df['n1']))
    df['chose_risky']   = np.where(risky_first, ~df['choice'], df['choice'])
    df['n_safe']        = np.where(risky_first, df['n2'], df['n1'])
    df['risky_first']   = risky_first
    df['order']         = np.where(risky_first, 'Risky first', 'Safe first')
    df['log_ratio_bin'] = (pd.cut(df['log_ratio'], bins=10)
                             .apply(lambda x: x.mid).astype(float))
    df['n_safe_bin']    = pd.qcut(df['n_safe'], q=3, labels=['Low stakes', 'Mid stakes', 'High stakes'])
    return df

df_dot_p = prep_df(df_dot)
df_sym_p = prep_df(df_sym)

## Presentation-order x stake-size interaction

Each panel shows P(chose risky) as a function of the log risky/safe magnitude ratio,
split by safe-option stake tertile.  The left column shows trials where the risky option
came first; the right column shows trials where the safe option came first.

The dashed vertical line marks the risk-neutral indifference point log(1/0.55).

In [ ]:
stake_pal = {'Low stakes': '#4C72B0', 'Mid stakes': '#DD8452', 'High stakes': '#55A868'}

def plot_interaction(df_p, axes_row, task_label):
    """Plot stake x order interaction, one axis per order condition."""
    for ax, order_val in zip(axes_row, ['Risky first', 'Safe first']):
        sub  = df_p[df_p['order'] == order_val]
        subj = (sub.groupby(['subject', 'log_ratio_bin', 'n_safe_bin'])['chose_risky']
                   .mean().reset_index())
        subj = subj[subj.groupby(['log_ratio_bin', 'n_safe_bin'])
                        ['subject'].transform('count') >= 3]
        hue_order = ['Low stakes', 'Mid stakes', 'High stakes']
        sns.lineplot(data=subj, x='log_ratio_bin', y='chose_risky',
                     hue='n_safe_bin', style='n_safe_bin',
                     hue_order=hue_order, style_order=hue_order,
                     palette=stake_pal, markers=True, dashes=False,
                     errorbar='se', ax=ax)
        ax.axhline(.5,            ls='--', c='gray', lw=1)
        ax.axvline(np.log(1/.55), ls='--', c='#333333', lw=1.5, label='Risk-neutral')
        ax.set_ylim(-.05, 1.05)
        ax.set_xlabel('log(risky / safe)')
        ax.set_ylabel('P(chose risky)')
        ax.set_title(f'{task_label} — {order_val}', fontsize=10)
        ax.legend(title='Safe stake', fontsize=8)
        sns.despine(ax=ax)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=True)
plot_interaction(df_dot_p, axes[0], 'Dot clouds (3T + 7T)')
plot_interaction(df_sym_p, axes[1], 'Symbolic (Arabic numerals)')
plt.suptitle('Presentation-order × safe-stake interaction', fontsize=13, y=1.01)
plt.tight_layout()

## Fit three models — dot-cloud data

Hierarchical MCMC, 100 draws / 100 tune / 2 chains.

In [ ]:
# ── 1. Expected Utility ──────────────────────────────────────────────────────
model_eu = ExpectedUtilityRiskModel(paradigm=df_dot)
model_eu.build_estimation_model(data=df_dot, hierarchical=True, save_p_choice=True)
idata_eu = model_eu.sample(draws=100, tune=100, chains=2, progressbar=True)

In [ ]:
# ── 2. KLW (shared noise, shared prior) ─────────────────────────────────────
model_klw = RiskModel(paradigm=df_dot, prior_estimate='klw',
                      fit_seperate_evidence_sd=False)
model_klw.build_estimation_model(data=df_dot, hierarchical=True, save_p_choice=True)
idata_klw = model_klw.sample(draws=100, tune=100, chains=2, progressbar=True)

In [ ]:
# ── 3. PMCM (separate noise + separate priors) ────────────────────
model_full = RiskModel(paradigm=df_dot, prior_estimate='full',
                       fit_seperate_evidence_sd=True)
model_full.build_estimation_model(data=df_dot, hierarchical=True, save_p_choice=True)
idata_full = model_full.sample(draws=100, tune=100, chains=2, progressbar=True)

## Posterior predictives — dot-cloud data

Dots = observed group average; line + shading = model mean and 95 % posterior interval.

In [ ]:
def add_model_ppc(df_orig, df_prepped, model, idata, model_name, n_ppc_samples=50):
    """Return df_prepped with added posterior-predictive columns."""
    ppc_df = model.ppc(df_orig, idata, var_names=['p'])
    ppc_p  = ppc_df.xs('p', level='variable')
    cols   = np.random.choice(ppc_p.columns,
                               size=min(n_ppc_samples, ppc_p.shape[1]),
                               replace=False)
    ppc_p  = ppc_p[cols]
    risky_first = df_prepped['risky_first'].values
    p_risky = np.where(risky_first[:, None], 1 - ppc_p.values, ppc_p.values)
    df_out = df_prepped.copy()
    df_out['p_mean'] = p_risky.mean(1)
    df_out['p_lo']   = np.percentile(p_risky, 2.5,  axis=1)
    df_out['p_hi']   = np.percentile(p_risky, 97.5, axis=1)
    df_out['model']  = model_name
    return df_out


def plot_ppc_interaction(df_pred, model_name, axes_row):
    hue_order = ['Low stakes', 'Mid stakes', 'High stakes']
    for ax, order_val in zip(axes_row, ['Risky first', 'Safe first']):
        sub  = df_pred[df_pred['order'] == order_val]
        obs  = sub.groupby(['n_safe_bin', 'log_ratio_bin'])['chose_risky'].mean().reset_index()
        pred = (sub.groupby(['n_safe_bin', 'log_ratio_bin'])[['p_mean', 'p_lo', 'p_hi']]
                   .mean().reset_index())
        for sbin in hue_order:
            o = obs[obs['n_safe_bin']  == sbin]
            p = pred[pred['n_safe_bin'] == sbin]
            if len(o) == 0:
                continue
            c = stake_pal[sbin]
            ax.fill_between(p['log_ratio_bin'], p['p_lo'], p['p_hi'],
                            color=c, alpha=.20)
            ax.plot(p['log_ratio_bin'], p['p_mean'], color=c, lw=2, label=sbin)
            ax.scatter(o['log_ratio_bin'], o['chose_risky'],
                       color=c, s=25, zorder=5, alpha=.85)
        ax.axhline(.5,            ls='--', c='gray', lw=1)
        ax.axvline(np.log(1/.55), ls='--', c='#333333', lw=1.5)
        ax.set_ylim(-.05, 1.05)
        ax.set_title(f'{model_name} — {order_val}', fontsize=9)
        ax.set_xlabel('log(risky / safe)')
        ax.set_ylabel('P(chose risky)')
        ax.legend(title='Safe stake', fontsize=7, loc='upper left')
        sns.despine(ax=ax)


fig, axes = plt.subplots(3, 2, figsize=(12, 13), sharey=True)
for (mdl, idat, name), row in zip(
        [(model_eu,   idata_eu,   'EU'),
         (model_klw,  idata_klw,  'KLW'),
         (model_full, idata_full, 'PMCM')],
        axes):
    df_pred = add_model_ppc(df_dot, df_dot_p, mdl, idat, name)
    plot_ppc_interaction(df_pred, name, row)

plt.suptitle('Posterior predictive checks — dot-cloud data  (dots = observed, shading = 95 % CI)',
             fontsize=11, y=1.01)
plt.tight_layout()

## Fit three models — symbolic data

In [ ]:
# ── 1. EU ────────────────────────────────────────────────────────────────────
model_eu_sym = ExpectedUtilityRiskModel(paradigm=df_sym)
model_eu_sym.build_estimation_model(data=df_sym, hierarchical=True, save_p_choice=True)
idata_eu_sym = model_eu_sym.sample(draws=100, tune=100, chains=2, progressbar=True)

In [ ]:
# ── 2. KLW ───────────────────────────────────────────────────────────────────
model_klw_sym = RiskModel(paradigm=df_sym, prior_estimate='klw',
                           fit_seperate_evidence_sd=False)
model_klw_sym.build_estimation_model(data=df_sym, hierarchical=True, save_p_choice=True)
idata_klw_sym = model_klw_sym.sample(draws=100, tune=100, chains=2, progressbar=True)

In [ ]:
# ── 3. PMCM ────────────────────────────────────────────────────────
model_full_sym = RiskModel(paradigm=df_sym, prior_estimate='full',
                            fit_seperate_evidence_sd=True)
model_full_sym.build_estimation_model(data=df_sym, hierarchical=True, save_p_choice=True)
idata_full_sym = model_full_sym.sample(draws=100, tune=100, chains=2, progressbar=True)

## Posterior predictives — symbolic data

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 13), sharey=True)
for (mdl, idat, name), row in zip(
        [(model_eu_sym,   idata_eu_sym,   'EU'),
         (model_klw_sym,  idata_klw_sym,  'KLW'),
         (model_full_sym, idata_full_sym, 'PMCM')],
        axes):
    df_pred = add_model_ppc(df_sym, df_sym_p, mdl, idat, name)
    plot_ppc_interaction(df_pred, name, row)

plt.suptitle('Posterior predictive checks — symbolic data  (dots = observed, shading = 95 % CI)',
             fontsize=11, y=1.01)
plt.tight_layout()

## Parameter interpretation: $\nu_1$ vs $\nu_2$

For the full-prior NLC model we extract subject-level posterior means for the two noise
parameters.  If the first-presented option accumulates extra noise, we expect $\nu_1 >
\nu_2$ and most subjects to lie above the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (idat, label) in zip(axes,
        [(idata_full,     'Dot clouds'),
         (idata_full_sym, 'Symbolic')]):
    nu1 = idat.posterior['n1_evidence_sd'].mean(('chain', 'draw')).values
    nu2 = idat.posterior['n2_evidence_sd'].mean(('chain', 'draw')).values
    lim = max(nu1.max(), nu2.max()) * 1.15
    ax.scatter(nu2, nu1, alpha=.7, s=45, color='#2166ac', zorder=3)
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='ν₁ = ν₂')
    ax.set_xlabel('Second-option noise  ν₂')
    ax.set_ylabel('First-option noise  ν₁')
    ax.set_title(label)
    ax.legend(fontsize=9); sns.despine(ax=ax)

plt.suptitle('Subject-level noise estimates from PMCM', fontsize=12, y=1.02)
plt.tight_layout()